# 01c - Does the localizer recipe v2 really work? (verification, version 2)
Run **after** notebook 01. It only loads what 01 saved. The decision rules are fixed **before** you look at any number.

**v2 counts as "working" only if all of these hold**
* **R1** at least 3 seeds of recipe v2 exist
* **R2** mean single-model hit@1 is at least **+0.05** above the original localizer
* **R3** the paired patient-level 95% CI of (v2 ensemble - original) hit@1 does not include 0
* **R4** the shuffled-mask control is at least **0.05 below** the v2 mean (otherwise the gain is not image evidence)
* **R5** sender side: lesion area lost at 30% loss is **not worse** (section 5)
* **R6** end-to-end AUC is **not worse**; whether it is *confirmed* (CI excludes 0) or only a *trend* decides how you may word the claim (section 6)

**Order:** setup, 1, 2 (about 2 h of GPU: 4 models), 3, 4, 5, 6, 7. Sections 1, 3, 4, 5 are quick. Nothing here changes the models you already have; new models get new names.

**What the author could not run:** training (section 2), the receiver and transmission code (sections 6, 7) and the setup cell, because the sandbox has no GPU, torch or your data. The logic of the other cells was tested on stand-in data.

In [ ]:
# ---- setup (same layout as notebook 01) ----
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Not in Colab:', e)
!pip -q install pydicom
import sys, os, numpy as np, pandas as pd, cv2, torch
from dataclasses import replace
CODE_DIR = next((c for c in ['/content/drive/MyDrive/mammo_semcom', '/content/mammo_semcom', '/content'] if os.path.exists(os.path.join(c, 'src', 'config.py'))), None)
assert CODE_DIR, 'code folder (with src/config.py) not found'
sys.path.insert(0, CODE_DIR)
from src.config import Cfg
from src import data, train as T
from src.channel import choose_lost, blocks_to_pixels
from src.pipeline import Receiver, diagnose_all, build_importance, run_transmission, summarize
from src.metrics import auc_safe, paired_bootstrap_diff, bootstrap_ci

cfg = Cfg(cbis_root='/content/drive/MyDrive/CBIS-DDSM', work_dir='/content/drive/MyDrive/mammo_semcom_work').ensure()
images, masks, images_df, abn_df = data.load_cache(cfg)
tr_idx = images_df[images_df.split == 'train'].img_idx.values
va_idx = images_df[images_df.split == 'val'].img_idx.values
te_df = images_df[(images_df.split == 'test') & (~images_df.mixed)].reset_index(drop=True)
X_te, M_te = images[te_df.img_idx.values], masks[te_df.img_idx.values]
pat, y_img = te_df.patient_id.values, te_df.label.values
def have(kind, name): return os.path.exists(cfg.path('models', f'{kind}_{name}.pt'))
mean_fn = lambda y, s: float(np.mean(s))                    # lets paired_bootstrap_diff compare any per-image score
dummy = np.zeros(len(te_df))

def mean_p(df, scheme, rate=0.3):
    """Per-image mean probability over the repeats of one (scheme, rate) condition of run_transmission."""
    return df[(df.scheme == scheme) & (df.rate == rate)].groupby('img').p.mean().sort_index().values
def loss_compare(dfs, base, new):
    for scheme in ['uniform', 'semantic']:
        d, lo, hi, p = paired_bootstrap_diff(auc_safe, y_img, mean_p(dfs[new], scheme), mean_p(dfs[base], scheme), pat, n=2000)
        print(f'  {scheme:8s} @30% loss: AUC diff ({new} - {base}) {d:+.3f}  95% CI ({lo:+.3f}, {hi:+.3f})  p={p:.3f}')
print(len(te_df), 'test images |', T.DEVICE)

## 1. Sanity checks: no leakage, and trivial baselines the localizer must clearly beat

In [ ]:
sp = {s: set(images_df[images_df.split == s].patient_id) for s in ['train', 'val', 'test']}
assert not (sp['train'] & sp['val']) and not (sp['train'] & sp['test']) and not (sp['val'] & sp['test']), 'PATIENT LEAKAGE between splits - STOP'
print('OK: no patient appears in two splits')

def chance_hit(masks_, imgs_, dilate=24, n=300, seed=0):
    """Hit rate of a random point on breast tissue (same 24 px rule as T.hit_rate)."""
    rng = np.random.default_rng(seed); k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dilate + 1,) * 2); r = []
    for m, a in zip(masks_, imgs_):
        d = cv2.dilate(m.astype(np.uint8), k); ys, xs = np.nonzero(T.to01(a) > 0.03); j = rng.integers(0, len(ys), n)
        r.append(d[ys[j], xs[j]].mean())
    return float(np.mean(r))

bright = [cv2.GaussianBlur(T.to01(a), (0, 0), 15) for a in X_te]
print('random tissue point :', round(chance_hit(M_te, X_te), 3))
print('brightest region    :', round(T.hit_rate(bright, M_te), 3))

## 2. Training: more seeds, the shuffled-mask control, and `robust_v2`
Four models, about 20-30 min each on a T4 (about 2 h in total). Each is saved as soon as it finishes and finished ones are skipped, so after a Colab disconnect just re-run the setup cell and this cell.
* `clean_v2_s2`, `clean_v2_s3`: more seeds of the same recipe (rule R1)
* `clean_v2_shuffled`: training masks assigned to the *wrong* images (validation masks stay correct). It shows what position priors alone give (rule R4). In its log the validation hit-rate must stay low; if it climbs to about 0.6, something is wrong.
* `robust_v2`: the same recipe trained with simulated packet loss, for the receiver in section 7

In [ ]:
RUN_EXTRA_SEEDS, RUN_SHUFFLED_CONTROL = True, True
cfg_v2 = replace(cfg, loc_loss='peak', loc_lambda=0.3, loc_warmup=5, loc_tversky=True, loc_soft_k=15,
                 loc_opt='adamw_split', loc_enc_lr_mult=0.3, loc_aug=True)          # identical to notebook 01
if RUN_EXTRA_SEEDS:
    for name, sd in [('clean_v2_s2', 44), ('clean_v2_s3', 45)]:
        if not have('localizer', name):
            T.train_localizer(replace(cfg_v2, seed=sd), images, masks, tr_idx, va_idx, robust=False, name=name)
if RUN_SHUFFLED_CONTROL and not have('localizer', 'clean_v2_shuffled'):
    ms = np.array(masks, copy=True); ms[tr_idx] = masks[np.random.default_rng(0).permutation(tr_idx)]
    T.train_localizer(cfg_v2, images, ms, tr_idx, va_idx, robust=False, name='clean_v2_shuffled')
if not have('localizer', 'robust_v2'):                                              # receiver localizer for the packet-loss experiments
    T.train_localizer(replace(cfg_v2, seed=42), images, masks, tr_idx, va_idx, robust=True, name='robust_v2')
print({n: have('localizer', n) for n in ['clean_v2', 'clean_v2_s1', 'clean_v2_s2', 'clean_v2_s3', 'clean_v2_shuffled', 'robust_v2']})

## 3. Hit rate with patient-level paired confidence intervals, and the verdict (R1-R4)

In [ ]:
def per_image_hits(heat, masks_, dilate=24):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dilate + 1,) * 2); out = []
    for h, m in zip(heat, masks_):
        cy, cx, _ = T.localize(h); d = cv2.dilate(m.astype(np.uint8), k); out.append(int(d[int(round(cy)), int(round(cx))]))
    return np.array(out, float)

names = [n for n in ['clean', 'clean_peak', 'clean_v2', 'clean_v2_s1', 'clean_v2_s2', 'clean_v2_s3'] if have('localizer', n)]
v2 = [n for n in names if n.startswith('clean_v2')]
HM = {n: T.predict_heatmaps(T.load_localizer(cfg, n), X_te) for n in names}          # float16 maps: keeps RAM low
ENS = f'v2 ensemble ({len(v2)})'; acc = np.zeros(HM['clean'].shape, np.float32)
for n in v2: acc += HM[n]
HM[ENS] = (acc / len(v2)).astype(np.float16); del acc
hits = {n: per_image_hits(h, M_te) for n, h in HM.items()}
rows = []
for n, h in HM.items():
    d, lo, hi, p = paired_bootstrap_diff(mean_fn, dummy, hits[n], hits['clean'], pat)
    rows.append(dict(model=n, hit_at_1=hits[n].mean(), hit_at_3=T.hit_rate_topk(h, M_te, 3), dice=T.dice_np(h, M_te),
                     diff_vs_original=d, ci_low=lo, ci_high=hi, p=p))
tab = pd.DataFrame(rows).round(3); tab.to_csv(cfg.path('results', 'verify_localizer.csv'), index=False); display(tab)

seed_hits = [hits[n].mean() for n in v2]
gain = float(np.mean(seed_hits) - hits['clean'].mean()); ci_lo = tab.loc[tab.model == ENS, 'ci_low'].iloc[0]
print(f'single-model v2 hit@1: mean {np.mean(seed_hits):.3f}, sd {np.std(seed_hits):.3f} over {len(v2)} seeds | gain vs original {gain:+.3f}')
print('R1  >=3 seeds                    :', 'PASS' if len(v2) >= 3 else 'FAIL (run section 2)')
print('R2  mean gain >= +0.05           :', 'PASS' if gain >= 0.05 else 'FAIL')
print('R3  ensemble CI excludes 0       :', 'PASS' if ci_lo > 0 else 'FAIL')
if have('localizer', 'clean_v2_shuffled'):
    ctrl = per_image_hits(T.predict_heatmaps(T.load_localizer(cfg, 'clean_v2_shuffled'), X_te), M_te).mean()
    print(f'shuffled-mask control hit@1 = {ctrl:.3f}')
    print('R4  control >= 0.05 below v2     :', 'PASS' if np.mean(seed_hits) - ctrl >= 0.05 else 'FAIL - gain may come from position priors, do not go further')
else:
    print('R4  control not trained yet (section 2)')

## 4. Where does it help? (lesion type and lesion size)

In [ ]:
calc_img = abn_df.groupby('img_idx').lesion_type.apply(lambda s: s.astype(str).str.lower().str.contains('calc').any())
area = np.array([(m > 0).sum() for m in M_te])
te = te_df.assign(calcification=te_df.img_idx.map(calc_img), size=pd.qcut(pd.Series(area).rank(method='first'), 3, labels=['small', 'medium', 'large']),
                  original=hits['clean'], v2_ensemble=hits[ENS])
print(te.groupby('calcification')[['original', 'v2_ensemble']].agg(['mean', 'count']).round(3))
print(te.groupby('size', observed=True)[['original', 'v2_ensemble']].mean().round(3))

## 5. Sender side (R5): does the better hit rate protect the lesion at least as well?
The sender turns the heat-map into block priorities with a fixed 0.5 threshold, and recipe v2 (Tversky + soft targets) changes the probability scale. So check the lesion area actually lost at 30% semantic loss (lower is better) and how many blocks get top priority (fewer = the protected set is not diluted). If v2 is clearly **worse** here, use the original localizer for the sender and v2 only in the receiver.

In [ ]:
def sender_check(heat, rate=0.3, reps=3):
    imp = build_importance(cfg, X_te, M_te, heat=heat, source='pred')
    n2 = np.array([(i == 2).sum() for i in imp], float); ll = np.zeros(len(imp))
    for r in range(reps):
        rng = np.random.default_rng(r)
        for k, m in enumerate(M_te):
            lost = choose_lost(imp[k], rate, 'semantic', rng, cfg.burst_len)
            g = m > 0; ll[k] += (blocks_to_pixels(lost, cfg.block, m.shape) & g).sum() / max(g.sum(), 1) / reps
    return ll, n2

ll0, n0 = sender_check(HM['clean']); ll1, n1 = sender_check(HM[ENS])
d, lo, hi, p = paired_bootstrap_diff(mean_fn, dummy, ll1, ll0, pat)
print(f'lesion area lost @30%: original {ll0.mean():.3f} | v2 {ll1.mean():.3f} | diff {d:+.3f} (95% CI {lo:+.3f}, {hi:+.3f})')
print(f'priority-2 blocks per image: original {n0.mean():.1f} | v2 {n1.mean():.1f}')
print('R5  sender side:', 'WORSE (CI above 0): keep the original localizer for the sender' if lo > 0 else 'BETTER' if hi < 0 else 'no clear difference (PASS: not worse)')

## 6. End to end (R6): does it improve diagnosis? Clean images, then 30% loss
Same receiver and the same `clean_*` classifiers; only the localizer changes (the sender uses the same localizer as the receiver in this section). Reported for every v2 seed and for the ensemble: AUC, the **paired** AUC difference to the original, and the paired Brier-score difference (negative = v2 better). Brier is a secondary metric chosen in advance because it is less noisy than AUC; report it whatever it shows. The last lines split AUC by whether the localizer hit the lesion.

In [ ]:
class AvgLoc(torch.nn.Module):
    """Averages the members' probabilities (what section 3 evaluated) and returns a logit so Receiver/predict_heatmaps work unchanged."""
    def __init__(self, ms): super().__init__(); self.ms = torch.nn.ModuleList(ms)
    def forward(self, x):
        p = sum(torch.sigmoid(m(x).float()) for m in self.ms) / len(self.ms)
        return torch.logit(p.clamp(1e-6, 1 - 1e-6))

clfs = [T.load_classifier(cfg, f'clean_{k}') for k in range(cfg.ensemble_size)]
img01 = np.stack([T.to01(a) for a in X_te])
locs = {'original': T.load_localizer(cfg, 'clean'), ENS: AvgLoc([T.load_localizer(cfg, n) for n in v2]).to(T.DEVICE)}
for n in v2: locs[n] = T.load_localizer(cfg, n)
P = {k: diagnose_all(Receiver(cfg, loc, clfs), img01)['p'] for k, loc in locs.items()}
brier = lambda p: (np.asarray(p) - y_img) ** 2
rows = []
for k, p in P.items():
    a, lo, hi = bootstrap_ci(auc_safe, y_img, p, pat, n=1000); row = dict(localizer=k, AUC=a, AUC_lo=lo, AUC_hi=hi)
    if k != 'original':
        d, dlo, dhi, dp = paired_bootstrap_diff(auc_safe, y_img, p, P['original'], pat, n=2000)
        b, blo, bhi, bp = paired_bootstrap_diff(mean_fn, dummy, brier(p), brier(P['original']), pat, n=2000)
        row.update(dAUC=d, dAUC_lo=dlo, dAUC_hi=dhi, dAUC_p=dp, dBrier=b, dBrier_lo=blo, dBrier_hi=bhi, dBrier_p=bp)
    rows.append(row)
res6 = pd.DataFrame(rows).round(3); res6.to_csv(cfg.path('results', 'verify_localizer_e2e_clean.csv'), index=False); display(res6)
d, lo = res6.loc[res6.localizer == ENS, ['dAUC', 'dAUC_lo']].iloc[0]
print('R6  end-to-end AUC, v2 ensemble vs original:', 'CONFIRMED improvement (CI excludes 0)' if lo > 0 else 'WORSE' if d < 0 else 'trend only (CI includes 0): word the claim as a trend')
for k in ['original', ENS]:
    h = hits['clean' if k == 'original' else ENS] > 0
    print(f'{k:22s} AUC when localizer hit {auc_safe(y_img[h], P[k][h]):.3f} (n={int(h.sum())}) | when missed {auc_safe(y_img[~h], P[k][~h]):.3f} (n={int((~h).sum())})')

sender_dfs = {}
for k in ['original', ENS]:
    heat = T.predict_heatmaps(locs[k], X_te)
    imp = {'none': build_importance(cfg, X_te, M_te, source='none'), 'pred': build_importance(cfg, X_te, M_te, heat=heat, source='pred')}
    sender_dfs[k] = run_transmission(cfg, X_te, M_te, te_df, {'std': Receiver(cfg, locs[k], clfs)}, imp,
                                     [('uniform', 'none'), ('semantic', 'pred')], rates=[0.3], reps=3, verbose=False)
res6b = pd.concat([summarize(df).assign(localizer=k) for k, df in sender_dfs.items()])[['localizer', 'scheme', 'source', 'rate', 'AUC', 'AUC_sd', 'loc_hit', 'lesion_loss']].round(3)
res6b.to_csv(cfg.path('results', 'verify_localizer_e2e_loss.csv'), index=False); display(res6b)
loss_compare(sender_dfs, 'original', ENS)

## 7. The packet-loss pair: `clean_v2` sender + `robust_v2` receiver vs the original `clean` + `robust`
Uses the robust classifiers (`robust_*`). If notebook 02 pairs sender and receiver localizers differently, change the `pairs` dictionary to match it.

In [ ]:
if not (have('localizer', 'robust_v2') and all(have('classifier', f'robust_{k}') for k in range(cfg.ensemble_size))):
    print('robust_v2 or robust_* classifiers missing: run section 2 / notebook 01 first')
else:
    rclfs = [T.load_classifier(cfg, f'robust_{k}') for k in range(cfg.ensemble_size)]
    pairs = {'original': ('clean', 'robust'), 'v2': ('clean_v2', 'robust_v2')}          # (sender localizer, receiver localizer)
    rdfs = {}
    for k, (sn, rn) in pairs.items():
        heat = T.predict_heatmaps(T.load_localizer(cfg, sn), X_te)
        imp = {'none': build_importance(cfg, X_te, M_te, source='none'), 'pred': build_importance(cfg, X_te, M_te, heat=heat, source='pred')}
        rdfs[k] = run_transmission(cfg, X_te, M_te, te_df, {'robust': Receiver(cfg, T.load_localizer(cfg, rn), rclfs)}, imp,
                                   [('uniform', 'none'), ('semantic', 'pred')], rates=[0.3], reps=3, verbose=False)
    res7 = pd.concat([summarize(df).assign(config=k) for k, df in rdfs.items()])[['config', 'scheme', 'source', 'rate', 'AUC', 'AUC_sd', 'loc_hit', 'lesion_loss']].round(3)
    res7.to_csv(cfg.path('results', 'verify_localizer_e2e_robust.csv'), index=False); display(res7)
    loss_compare(rdfs, 'original', 'v2')

## What you may write, decided in advance
* **R1-R4 pass:** "Recipe v2 raised localization hit@1 from X to Y over N seeds (paired patient-level 95% CI ...); a shuffled-mask control reaches only Z."
* **R4 fails:** do not claim image-based improvement; report the control honestly and stop.
* **R5 worse:** "the better hit rate did not translate into better sender-side protection; the original localizer is kept for the sender."
* **R6 confirmed (CI excludes 0):** "improves end-to-end AUC by +0.0xx (95% CI ...)". **R6 trend only:** "end-to-end AUC moved in the same direction (+0.0xx) but the paired CI includes 0 on 453 test images, so the gain is not statistically confirmed."
* Always add: the recipe was designed while looking at test-set hit rates, so the new seeds and the control (trained after that) are the cleanest evidence. Report every seed, not the best one.